# 분류 연습문제 2 — LightGBM · XGBoost · 보팅

ESAA YB · 2026-11-06 세션

DACON 와인 품질 분류 데이터로 세 가지를 실습한다.

1. **LightGBM** — 데이터를 직접 정형해서 학습하고 제출 파일 만들기
2. **XGBoost** — 원핫 인코딩을 써서 같은 흐름 반복하기
3. **보팅** — 세 모델을 `VotingClassifier`로 묶기

# 1. LightGBM

* 데이터 '와인품질분류.zip'입니다.

1. 필요한 라이브러리를 가져옵니다.

In [1]:
# 세션에서는 와인품질분류.zip을 직접 업로드했지만,
# 다시 돌릴 때를 위해 같은 데이터를 내려받는 셀을 넣어 둔다.
import os
if not os.path.exists('train.csv'):
    import urllib.request, zipfile
    urllib.request.urlretrieve('https://bit.ly/3i4n1QB', 'wine.zip')
    with zipfile.ZipFile('wine.zip') as z:
        z.extractall('.')
print(sorted(f for f in os.listdir('.') if f.endswith('.csv')))

['sample_submission.csv', 'test.csv', 'train.csv']


In [2]:
import pandas as pd
import lightgbm as lgbm

2. 데이터를 불러옵니다.

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
smpl_sub = pd.read_csv('sample_submission.csv')

In [4]:
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,white
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,red
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,white
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,white
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,white


In [5]:
test.head()

,index,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,white
1,1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,red
2,2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,white
3,3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,white
4,4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,white


3. 변수를 변환합니다.

`type`은 white / red 두 가지 문자열이다. 트리 모델도 문자열은 그대로 받지 못하므로 숫자로 바꾼다.
**train과 test에 같은 규칙을 적용해야 한다.** 한쪽만 바꾸거나 서로 다른 숫자를 주면 모델이 다른 뜻으로 읽는다.

In [6]:
# type에는 white와 red 두 종류가 있습니다.
# 각각 0,1로 변환합니다.
train['type'] = train['type'].map({'white': 0, 'red': 1})
test['type'] = test['type'].map({'white': 0, 'red': 1})

print(train['type'].value_counts())
print(test['type'].value_counts())

type
0    4159
1    1338
Name: count, dtype: int64
type
0    739
1    261
Name: count, dtype: int64


In [7]:
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,0
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,1
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,0
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,0
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,0


In [8]:
test.head()

,index,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,0
1,1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,1
2,2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,0
3,3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,0
4,4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,0


In [9]:
# 모델에 입력하기 전 데이터를 정형합니다.
# train_x : train의 'index', 'quality' 피처를 제거하고 저장합니다.
# train_y : train의 'quality'를 저장합니다.
# test_x : test의 'index'를 제거하고 저장합니다.

train_x = train.drop(['index', 'quality'], axis=1)
train_y = train['quality']
test_x = test.drop('index', axis=1)

In [10]:
train_x.shape, train_y.shape, test_x.shape

((5497, 12), (5497,), (1000, 12))

In [11]:
train_x.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,0
1,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,1
2,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,0
3,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,0
4,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,0


In [12]:
train_y.head()

0    5
1    5
2    5
3    6
4    6
Name: quality, dtype: int64

In [13]:
test_x.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,0
1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,1
2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,0
3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,0
4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,0


`train_x`와 `test_x`의 열이 12개로 같다. 열 개수나 순서가 다르면 예측이 어긋나므로 여기서 확인하고 넘어간다.

4. LightGBM 모델을 생성 및 훈련합니다.

In [14]:
lgbm_model = lgbm.LGBMClassifier(random_state=42, verbose=-1)
lgbm_model.fit(train_x, train_y)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


5. 생성한 모델로 예측 데이터를 생성합니다.

In [15]:
lgbm_pred = lgbm_model.predict(test_x)

print(lgbm_pred[:20])
print('예측 개수:', len(lgbm_pred))
print(pd.Series(lgbm_pred).value_counts().sort_index())

[5 5 6 5 6 6 6 6 6 7 6 5 7 6 6 5 6 6 6 6]
예측 개수: 1000
4     10
5    355
6    490
7    134
8     11
Name: count, dtype: int64


6. 제출파일을 생성합니다.

문제의 `head(smpl_sub)`는 그대로 쓰면 `NameError`가 난다. 파이썬에 `head()`라는 내장 함수는 없고,
`head()`는 DataFrame의 **메서드**라서 `smpl_sub.head()`로 불러야 한다.

In [16]:
smpl_sub.head()

,index,quality
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [17]:
# 서브미션 파일의 quality 변수에 예측값을 대입합니다.
smpl_sub['quality'] = lgbm_pred

In [18]:
smpl_sub.head()

,index,quality
0,0,5
1,1,5
2,2,6
3,3,5
4,4,6


In [19]:
# csv 파일로 저장합니다.
smpl_sub.to_csv('submission.csv', index=False)

print(pd.read_csv('submission.csv').head())

   index  quality
0      0        5
1      1        5
2      2        6
3      3        5
4      4        6


# 2. XGBoost

1번과 같은 데이터인데 두 가지가 다르다.

- `type`을 0/1로 직접 바꾸는 대신 **원핫 인코딩**(`pd.get_dummies()`)을 쓴다.
- XGBoost는 **정답 라벨이 0부터 시작해야 한다.** 품질은 3~9점이라 그대로 넣으면 오류가 난다.

In [20]:
# 데이터 다운로드 링크로 데이터를 코랩에 불러옵니다.

!wget -q -O 3i4n1QB 'https://bit.ly/3i4n1QB'

import zipfile, os, urllib.request
if not os.path.exists('3i4n1QB'):          # wget이 없는 환경 대비
    urllib.request.urlretrieve('https://bit.ly/3i4n1QB', '3i4n1QB')

with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

print(sorted(os.listdir('data')))

['sample_submission.csv', 'test.csv', 'train.csv']


In [21]:
# 라이브러리 및 데이터 불러오기
import pandas as pd
from xgboost import XGBClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.drop('index', axis=1, inplace=True)
test.drop('index', axis=1, inplace=True)

In [22]:
# 데이터 확인
print(train.shape, test.shape)
print()
print(train.dtypes)
print()
print('결측값:', train.isnull().sum().sum())
train.head()

(5497, 13) (1000, 12)

quality                   int64
fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
type                        str
dtype: object

결측값: 0


,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,white
1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,red
2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,white
3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,white
4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,white


### 원핫 인코딩

`pd.get_dummies()`는 문자열 열을 찾아 값마다 열을 하나씩 만든다.
`type` 하나가 `type_red`, `type_white` 두 열로 바뀐다.

1번에서는 0/1 숫자 하나로 바꿨는데, 결과는 거의 같다. 범주가 두 개뿐이면
"red인가 아닌가" 한 열이면 충분하고, 원핫은 그 정보를 두 열에 나눠 담았을 뿐이다.

In [23]:
# 원핫 인코딩 (pd.get_dummies())
train = pd.get_dummies(train)
test = pd.get_dummies(test)

print(train.columns.tolist())
print(test.columns.tolist())

['quality', 'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'type_red', 'type_white']
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'type_red', 'type_white']


`train`에만 `quality`가 더 있고 나머지 열은 같다. 순서까지 같은지 뒤에서 한 번 더 맞춘다.

In [24]:
# 모델 정의
xgb_model = XGBClassifier(random_state=42)

In [25]:
# 모델 학습
# X 는 train에서 quality 를 제외한 모든 변수
# y 는 train의 qulity 변수

X = train.drop('quality', axis=1)
y = train['quality']

# XGBoost는 라벨이 0부터 시작해야 한다. 품질 3~9점을 0~6으로 옮긴다.
y_shift = y - 3

# fit 메소드를 이용해 모델 학습
xgb_model.fit(X, y_shift)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklear

In [26]:
# predict 메소드와 원핫 인코딩한 test 데이터를 이용해 품질 예측

# 열 순서를 train과 똑같이 맞춘다
test = test[X.columns]

xgb_pred = xgb_model.predict(test) + 3   # 0~6으로 나온 예측을 다시 3~9로 되돌린다

print(xgb_pred[:20])
print(pd.Series(xgb_pred).value_counts().sort_index())

[5 5 6 5 6 6 6 6 6 7 6 6 7 6 5 5 6 6 6 5]
4      8
5    355
6    497
7    126
8     14
Name: count, dtype: int64


In [27]:
# sample_submission.csv 파일을 불러와 예측된 값으로 채워 주기
submission = pd.read_csv('data/sample_submission.csv')
submission['quality'] = xgb_pred
submission.head()

,index,quality
0,0,5
1,1,5
2,2,6
3,3,5
4,4,6


In [28]:
# 정답파일 내보내기
submission.to_csv('xgb_pred.csv', index=False)

print(pd.read_csv('xgb_pred.csv').head())

   index  quality
0      0        5
1      1        5
2      2        6
3      3        5
4      4        6


# 3. 보팅

LightGBM, XGBoost, 랜덤포레스트 세 모델을 `VotingClassifier`로 묶는다.

`voting='soft'`는 세 모델이 내놓은 **확률을 평균**해서 가장 높은 등급을 고른다.
`voting='hard'`는 각 모델의 예측을 한 표씩 세어 다수결로 정한다.
보통 확률 정보를 버리지 않는 소프트 보팅이 조금 낫다.

In [29]:
# 데이터 다운로드 링크로 데이터를 코랩에 불러옵니다.

!wget -q -O 3i4n1QB 'https://bit.ly/3i4n1QB'

import zipfile, os, urllib.request
if not os.path.exists('3i4n1QB'):          # wget이 없는 환경 대비
    urllib.request.urlretrieve('https://bit.ly/3i4n1QB', '3i4n1QB')

with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

print(sorted(os.listdir('data')))

['sample_submission.csv', 'test.csv', 'train.csv']


In [30]:
# 라이브러리 및 데이터 불러오기
import pandas as pd
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.drop('index', axis=1, inplace=True)
test.drop('index', axis=1, inplace=True)

In [31]:
# 데이터 확인
print(train.shape, test.shape)
print(train['quality'].value_counts().sort_index())

(5497, 13) (1000, 12)
quality
3      26
4     186
5    1788
6    2416
7     924
8     152
9       5
Name: count, dtype: int64


In [32]:
# 원핫 인코딩 (pd.get_dummies())
train = pd.get_dummies(train)
test = pd.get_dummies(test)

print(train.shape, test.shape)

(5497, 14) (1000, 13)


In [33]:
# 모델 정의 (LightGBM, XGBoost, Random Forest)
lgbm_clf = LGBMClassifier(random_state=42, verbose=-1)
xgb_clf = XGBClassifier(random_state=42)
rf_clf = RandomForestClassifier(random_state=42)

# VotingClassifier 정의
vo_clf = VotingClassifier(
    estimators=[('LGBM', lgbm_clf), ('XGB', xgb_clf), ('RF', rf_clf)],
    voting='soft')

In [34]:
# 모델 학습

# X 는 train에서 quality 를 제외한 모든 변수
# y 는 train의 qulity 변수

X = train.drop('quality', axis=1)
y = train['quality']

# XGBoost가 섞여 있으므로 라벨을 0부터 시작하도록 옮긴다
y_shift = y - 3

# fit 메소드를 이용해 모델 학습
vo_clf.fit(X, y_shift)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('LGBM', ...), ('XGB', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](7,)","[0,1,2,...,4,5,6]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[LGBMClassifie...2, verbose=-1), XGBClassifier...ree=None, ...), RandomForestC...ndom_state=42)]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['fixed acidity','volatile acidity','citric acid',...,'alcohol','type_red', 'type_white']"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()


In [35]:
# predict 메소드와 원핫 인코딩한 테스트 데이터를 이용해 품질 예측
test = test[X.columns]

vo_pred = vo_clf.predict(test) + 3

print(vo_pred[:20])
print(pd.Series(vo_pred).value_counts().sort_index())

[5 5 6 5 6 6 6 6 6 7 6 6 7 6 6 5 6 6 6 6]
4      8
5    358
6    494
7    128
8     12
Name: count, dtype: int64


In [36]:
submission = pd.read_csv('data/sample_submission.csv')
submission['quality'] = vo_pred
submission.to_csv('voting_pred.csv', index=False)
submission.head()

,index,quality
0,0,5
1,1,5
2,2,6
3,3,5
4,4,6


## 세 방법 비교

제출 파일만 만들면 어느 쪽이 나은지 알 수 없다. 정답이 있는 train에서 교차검증으로 비교한다.

In [37]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=42)

models = [
    ('LightGBM', LGBMClassifier(random_state=42, verbose=-1)),
    ('XGBoost', XGBClassifier(random_state=42)),
    ('랜덤포레스트', RandomForestClassifier(random_state=42)),
    ('소프트 보팅', VotingClassifier(
        estimators=[('LGBM', LGBMClassifier(random_state=42, verbose=-1)),
                    ('XGB', XGBClassifier(random_state=42)),
                    ('RF', RandomForestClassifier(random_state=42))],
        voting='soft')),
]

for name, m in models:
    s = cross_val_score(m, X, y_shift, cv=cv, n_jobs=1)
    print('%-12s %.4f (±%.4f)' % (name, s.mean(), s.std()))

print()
print('기준선 (전부 6점): %.4f' % (y == 6).mean())

LightGBM     0.6438 (±0.0146)


XGBoost      0.6507 (±0.0143)


랜덤포레스트       0.6705 (±0.0096)


소프트 보팅       0.6611 (±0.0122)

기준선 (전부 6점): 0.4395


## 정리

**1. `type`은 꼭 숫자로 바꿔야 한다.** 0/1 매핑이든 원핫 인코딩이든 결과는 비슷하다.
범주가 두 개뿐이면 열 하나로도 충분한 정보라서 그렇다. **train과 test에 같은 규칙을 적용하고,
열 순서까지 맞춰야** 예측이 어긋나지 않는다.

**2. XGBoost는 라벨이 0부터 시작해야 한다.** 품질 3~9점을 그대로 넣으면
`Invalid classes inferred from unique values of y` 오류가 난다. `y - 3`으로 옮겨 학습하고
예측 뒤에 `+ 3`으로 되돌린다. LightGBM과 랜덤포레스트는 이 제약이 없지만,
보팅으로 묶을 때는 XGBoost에 맞춰 전부 옮겨야 한다.

**3. 보팅이 항상 이기지는 않는다.** 세 모델을 묶었는데 단일 모델보다 낮게 나올 수 있다.
소프트 보팅은 확률을 평균하므로, 성능이 떨어지는 모델이 섞이면 그쪽이 평균을 끌어내린다.
**성능이 비슷한 모델끼리 묶을 때** 효과가 난다.

**4. `head(smpl_sub)`는 `smpl_sub.head()`로 고쳐야 한다.** `head()`는 내장 함수가 아니라
DataFrame의 메서드다.